## Time series at a single point with earthkit

This notebook shows how to extract an hourly time series from the Destination Earth Climate DT at a single location (Brussels) and visualise it with `earthkit.plots`.

**What you will learn**
- How to use Polytope's `timeseries` feature to retrieve hourly data efficiently at a single lat/lon
- How to compute temporal statistics with `earthkit.transforms`
- How to build multi-layer time series charts - lines, filled ranges, and multi-model overlays - using the `TimeSeries` class.

## Setup

This notebook uses the following components of earthkit:

<div align="center">
  <br>
  <a href="https://earthkit-data.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-data-light.svg" alt="earthkit-data" width="200">
  </a>
  <a href="https://earthkit-transforms.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-transforms-light.svg" alt="earthkit-transforms" width="200">
  </a>
  <a href="https://earthkit-plots.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-plots-light.svg" alt="earthkit-plots" width="200">
  </a>
</div>

In [ ]:
import earthkit.data as ekd
import earthkit.plots as ekp
from earthkit.plots.temporal import TimeSeries
import earthkit.transforms as ekt

### Define the target location

We pin the coordinates for Brussels once so both the Polytope request and any downstream labels stay consistent.

In [ ]:
LOCATION = [50.85, 4.35]

### Retrieve an hourly time series from the Climate DT

Polytope's `timeseries` feature type does server-side extraction at a single point, so only the values for that location are transferred rather than a full global field for every time step. This makes retrieving for example a year of hourly data very fast and lightweight.

The `feature` dictionary sets `type: timeseries` and passes the target coordinates. The `time_axis: date` option tells Polytope to organise the result along the date dimension.

In [ ]:
ifs_nemo_request = {
    "activity": "projections",   # CMIP activity: SSP scenario projections
    "class": "d1",               # DestinE data class
    "dataset": "climate-dt",     # dataset family
    "experiment": "ssp3-7.0",    # experiment/scenario (middle-of-the-road high emissions)
    "generation": "2",           # dataset generation
    "levtype": "sfc",            # vertical level type: surface fields
    "date": "20400101/to/20401231",  # date range: full year 2040 (Polytope range syntax)
    "model": "ifs-nemo",         # coupled atmosphere-ocean model
    "expver": "0001",            # experiment version (operations / research exp.)
    "param": "167",              # parameter ID: 2 m temperature
    "realization": "1",          # initial-condition run
    "resolution": "standard",    # quick-view ('standard') or full ('high') resolution
    "stream": "clte",            # climate hourly time-step stream
    "type": "fc",                # forecast fields
    "time": "0000/to/2300",      # all hours of the day
    "feature": {                 # server-side extraction at a single point
        "type" : "timeseries",   # feature type: extract a time series
        "points": [[LOCATION[0], LOCATION[1]]],  # target [lat, lon]
        "time_axis": "date",     # organise the result along the date dimension
    }
}

ifs_nemo = ekd.from_source(
    "polytope",
    "destination-earth",
    ifs_nemo_request,
    address="polytope.mn5.apps.dte.destination-earth.eu",
    stream=False,
).to_xarray()

### Inspect the Dataset

The result is an xarray Dataset with a single `t` time dimension covering all 8784 hourly time steps of 2040 (a leap year). The `latitude` and `longitude` coordinates reflect the nearest HEALPix cell centre to the requested point.

In [ ]:
ifs_nemo

### Hourly line plot

The `TimeSeries` class in earthkit-plots has many built-in features to support quick and easy plotting of timeseries data. Here we plot the raw hourly series and group the x-axis ticks by month with `frequency="M"` and `period=True`. The title template uses `{model!u}` to uppercase the model name and `{location:%c}, {location:%C}` to format the city and country.

In [ ]:
chart = TimeSeries()

chart.line(ifs_nemo, units="celsius")
chart.ylabel("{units}")
chart.title("{model!u} 2m temperature at {location:%c}, {location:%C} ({latitude:%Lt}, {longitude:%Ln})")
chart.xticks(frequency="M", period=True)
chart.show()

### Monthly statistics

The hourly series is noisy at the annual scale. `earthkit.transforms` provides temporal aggregation functions that work directly on xarray Datasets. We compute three monthly statistics in one step:

- `monthly_mean` — the mean temperature for each calendar month
- `monthly_min` — the coldest hourly reading in each month
- `monthly_max` — the warmest hourly reading in each month

These will be used to draw a mean line overlaid on a shaded min–max range.

In [ ]:
monthly_mean = ekt.temporal.monthly_mean(ifs_nemo)
monthly_min = ekt.temporal.monthly_min(ifs_nemo)
monthly_max = ekt.temporal.monthly_max(ifs_nemo)

### Monthly mean with min–max range

`chart.fill_between` shades the area between two datasets — here the monthly min and max — giving an immediate sense of intra-month variability. The mean line is drawn on top. `time_offset="15D"` shifts both layers to the middle of each month so the markers sit on the period they represent rather than at the start.

In [ ]:
chart = TimeSeries()

chart.fill_between(monthly_min, monthly_max, color="red", label="monthly range", time_offset="15D", units="celsius")
chart.line(monthly_mean, color="red", time_offset="15D", label="monthly mean", units="celsius")
chart.ylabel("{units}")
chart.title("{model!u} monthly mean 2m temperature at {location:%c}, {location:%C} ({latitude:%Lt}, {longitude:%Ln})")
chart.xticks(frequency="M", period=True)
chart.legend()
chart.show()

### Multi-model comparison

One of the strengths of the Climate DT is that multiple models are run for the same scenario. We can retrieve the same 2040 hourly series for the **ICON** and **IFS-FESOM** models by copying the original request dict and swapping the `model` key - all other fields stay identical.

Note that ICON and IFS-FESOM are served from the LUMI endpoint rather than MN5, requiring a small change to the `address` key.

In [ ]:
icon_request = ifs_nemo_request.copy()
icon_request["model"] = "icon"

icon = ekd.from_source(
    "polytope",
    "destination-earth",
    icon_request,
    address="polytope.lumi.apps.dte.destination-earth.eu",
    stream=False,
).to_xarray()

ifs_fesom_request = ifs_nemo_request.copy()
ifs_fesom_request["model"] = "ifs-fesom"

ifs_fesom = ekd.from_source(
    "polytope",
    "destination-earth",
    ifs_fesom_request,
    address="polytope.lumi.apps.dte.destination-earth.eu",
    stream=False,
).to_xarray()

### Overlay all three models

We loop over the three datasets and call `chart.line` once per model, adding each as a separate layer on the same axes. `drawstyle="spline"` smooths the daily mean lines so the seasonal signal is easy to compare across models. The `{model!u}` label placeholder is resolved per-dataset, so the legend entries are named automatically.

In [ ]:
chart = TimeSeries()

COLORS = ["cornflowerblue", "crimson", "goldenrod"]
MODELS = [ifs_nemo, icon, ifs_fesom]

for color, data in zip(COLORS, MODELS):
    chart.line(
        ekt.temporal.daily_mean(data),
        label="{model!u}",
        color=color,
        units="celsius",
        drawstyle="spline",
    )

chart.ylabel("{units}")
chart.title("{model!u} 2m temperature at {location:%c}, {location:%C} ({latitude:%Lt}, {longitude:%Ln})")
chart.xticks(frequency="M", period=True)
chart.legend()
chart.show()